#Copernicus data import and processing for the 2022 Environmental Oder Disaster

### Used imports

In [10]:
from scripts.copernicus_api import get_access_token, search_tile, download_product
from scripts.file_aux import process_resolution_for_aoi
from pathlib import Path
import os
import re
from zipfile import ZipFile
import shutil
import geopandas as gpd

### Area of interest

![alt text](figures/AOIs.png "Area of interest around Oder with tiles found by Copernicus. Some corner cases have been discarded.")

Area of interest around Oder with tiles found by Copernicus. Some corner cases have been discarded. The tiles are separated into two groups: lower Oder below Ścinawa and upper Oder up to Gliwice (images include farther parts).

We are mainly interested in three areas:
 - middle section of Oder around Krosno Odrzańskie with registered cholorphyll peak [(Stelzer, 2022)](https://www.igb-berlin.de/en/news/new-analysis-satellite-data-confirm-massive-algal-bloom-oder-river) (tile `T33UWT`),
 - Oder mouth around Świnoujście and Szczecin Lagoon (tile `T33UVV`),
 - Pomeranian Bay (tile `T33UVA`).

The chosen tiles are:
 - `T33UVA` - Pomeranian Bay (northernmost)
 - `T33UVV` - Szczecin and Szczecin Lagoon (right south of `T33UVA`)
 - `T33UVU` - Frankfurt to Gryfino segment (right south of `T33UVV`)
 - `T33UVT` - Lusatia, Krosno to Frankfurt segment (right south of `T33UVU`)
 - `T33UWT` - Ścinawa to Krosno segment (right east of `T33UVT`)
 - `T33UXS` - Krapkowice to Ścinawa segment (right south-east to `T33UWT`)
 - `T33UYR` - Upper Silesia, source to Krapkowice segment (right south-east to `T33UXS`)

In [11]:
lower_Oder_tiles = ["T33UVV", "T33UVU", "T33UVA", "T33UVT", "T33UWT"]
upper_Oder_tiles = ["T33UWS", "T33UYR", "T33UXS"]
tiles = ["T33UVT", "T33UWT"] # for testing only a single tile is taken

###Dates of interest

In [12]:
dates = [
    "2022-08-16",           # Little cloud cover over lower Oder around the peak
]


In [13]:
download_dir = Path("./raw_data")

print("Requesting Copernicus access token...")
token = get_access_token()

all_products = []

for tile in tiles:

    print(f"\nSearching tile {tile}...")

    products = search_tile(tile, token, dates[0])

    if not products:
        print("  No matching Sentinel-2 L2A products found.")
        continue

    for product in products:

        print(
            f"  Found: {product['Name']}\n"
            f"  Sensing start: {product['ContentDate']['Start']}\n"
            f"  Product ID: {product['Id']}\n"
        )

        all_products.append(product)

# Remove duplicates, if any.
unique_products = {
    product["Id"]: product
    for product in all_products
}

print(f"\nTotal unique products found: {len(unique_products)}")

for product in unique_products.values():
    download_product(product, token, download_dir)

print("\nFinished.")

Requesting Copernicus access token...

Searching tile T33UVT...
  Found: S2A_MSIL2A_20220816T100611_N0510_R022_T33UVT_20240710T182750.SAFE
  Sensing start: 2022-08-16T10:06:11.024000Z
  Product ID: d50063c9-80ad-4efb-a17d-b8ff057e436c


Searching tile T33UWT...
  Found: S2A_MSIL2A_20220816T100611_N0510_R022_T33UWT_20240710T182750.SAFE
  Sensing start: 2022-08-16T10:06:11.024000Z
  Product ID: fdc0c5aa-783c-4b8a-8d7d-d71baaeeafeb


Total unique products found: 2
Already downloaded — skipped:
  S2A_MSIL2A_20220816T100611_N0510_R022_T33UVT_20240710T182750.SAFE.zip
Already downloaded — skipped:
  S2A_MSIL2A_20220816T100611_N0510_R022_T33UWT_20240710T182750.SAFE.zip

Finished.


### Extract bands of interest

In [ ]:
files = os.listdir("raw_data")

zips = []
full_bands = ["B02_10m", "B03_10m", "B04_10m", "B05_20m", "B08_10m", "B11_20m", "SCL_20m"]
bands10 = ["B02", "B03", "B04", "B08"]
bands20 = ["B05", "B11", "SCL"]

for file in files:
    match = re.match(
        r"S2A_MSIL2A_(\d+)T(?:\d+)_(?:N\d+)_(?:R\d+)_(T\d\d\w\w\w)_(?:\d+T\d+)\.SAFE\.zip",
        file)
    if match:
        zips.append((match.group(1), match.group(2), file))

for (date, tile, filename) in zips:
    temp_dir = f"raw_data/tmp/{date}_{tile}"
    if not os.path.exists(temp_dir):
        with ZipFile(f"raw_data/{filename}", 'r') as zip_ref:
            zip_ref.extractall(temp_dir)
    data_dir = f"{temp_dir}/{filename[:-4]}/GRANULE"
    data_dir += f"/{os.listdir(data_dir)[0]}/IMG_DATA"
    data10_dir = f"{data_dir}/R10m"
    data20_dir = f"{data_dir}/R20m"
    datafiles10 = os.listdir(data10_dir)
    datafiles20 = os.listdir(data20_dir)
    os.makedirs(f"raw_data/extracted/{date}_{tile}/R10m", parents=True, exist_ok=True)
    os.makedirs(f"raw_data/extracted/{date}_{tile}/R20m", parents=True, exist_ok=True)
    for file in datafiles10:
        band = file[-11:-8]
        if band in bands10:
            shutil.copy(f"{data10_dir}/{file}", f"raw_data/extracted/{date}_{tile}/R10m/{band}_10m.jp2")
    for file in datafiles20:
        band = file[-11:-8]
        if band in bands20:
            shutil.copy(f"{data20_dir}/{file}", f"raw_data/extracted/{date}_{tile}/R20m/{band}_20m.jp2")
    shutil.rmtree(temp_dir)


In [ ]:
aoi_dir = Path("./raw_data/AOI/small_AOI.geojson")
band_dir = Path("./raw_data/extracted")
target_crs = "EPSG:3035"
aoi_id_field = "NAME"

resolutions = {
    "10m": 10,
    "20m": 20
}

nodata_value = 0

data_dirs = os.listdir(band_dir)
data_by_dates = {}
band_files = {}
for dir in data_dirs:
    date = dir.split("_")[0]
    if date not in data_by_dates:
        data_by_dates[date] = []
        band_files[date] = {band.split("_")[0]: [] for band in full_bands}
    data_by_dates[date].append(dir)
    for band in full_bands:
        band_file = Path(band_dir) / dir / f"R{band[4:]}" / f"{band}.jp2"
        if band_file.exists():
            band_files[date][band.split("_")[0]].append(band_file)


### Save the masked areas of interest

In [ ]:
masked_dir = Path("./local/masked/")
masked_dir.mkdir(parents=True, exist_ok=True)

aois = gpd.read_file("./raw_data/AOI/small_AOI.geojson")

aois["geometry"] = aois.geometry.buffer(0)
aois = aois.to_crs(target_crs)

aois = aois.dissolve(by=aoi_id_field, as_index=False)

for date_text, tile_folders in sorted(data_by_dates.items()):
    for _, aoi_row in aois.iterrows():

        aoi_name = aoi_row[aoi_id_field]
        aoi_geometry = aoi_row.geometry

        process_resolution_for_aoi(
            date_text=date_text,
            band_files=band_files,
            aoi_name=aoi_name,
            aoi_geometry=aoi_geometry,
            resolution_label="10m",
            bands=bands10,
            output_root=masked_dir
        )

        process_resolution_for_aoi(
            date_text=date_text,
            band_files=band_files,
            aoi_name=aoi_name,
            aoi_geometry=aoi_geometry,
            resolution_label="20m",
            bands=bands20,
            output_root=masked_dir
        )

